# Physics Parameter Exploration

Interactive exploration of the tokamak physics model.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from src.physics import derivedParams, stability_summary

plt.style.use('dark_background')
plt.rcParams.update({'figure.facecolor': '#060b14', 'axes.facecolor': '#020409',
                     'axes.edgecolor': '#1a2840', 'grid.color': '#1a2840'})
print('Ready')

## Safety factor q vs Ip and Bt

In [ ]:
Ip_range = np.linspace(0.5, 5.0, 50)
Bt_vals  = [2.0, 3.5, 5.0, 7.0]

fig, ax = plt.subplots(figsize=(9, 5))
for Bt in Bt_vals:
    q_vals = [derivedParams(ip, Bt, 20, 4.0).q for ip in Ip_range]
    ax.plot(Ip_range, q_vals, lw=2, label=f'Bt = {Bt} T')

ax.axhline(2.0, color='#ff3355', lw=1.2, ls='--', label='KS disruption limit (q=2)')
ax.axhline(3.5, color='#f5c842', lw=1.0, ls=':', label='ITER target q≈3.5')
ax.set_xlabel('Plasma current Ip (MA)', color='#c8d0e0')
ax.set_ylabel('Safety factor q', color='#c8d0e0')
ax.set_title('Safety factor vs plasma current', color='#00d4ff')
ax.legend(facecolor='#0b1525', edgecolor='#1a2840', labelcolor='#c8d0e0')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## β_N operating space

In [ ]:
Pnbi_range = np.linspace(0, 100, 60)
ne_vals    = [2.0, 4.0, 6.0, 10.0]

fig, ax = plt.subplots(figsize=(9, 5))
for ne in ne_vals:
    b_vals = [derivedParams(2.0, 3.5, p, ne).beta_N for p in Pnbi_range]
    ax.plot(Pnbi_range, b_vals, lw=2, label=f'ne = {ne}×10¹⁹ m⁻³')

ax.axhline(3.5, color='#ff3355', lw=1.2, ls='--', label='Troyon limit (β_N=3.5)')
ax.set_xlabel('NBI heating power P_NBI (MW)', color='#c8d0e0')
ax.set_ylabel('Normalised beta β_N', color='#c8d0e0')
ax.set_title('β_N operating space', color='#00d4ff')
ax.legend(facecolor='#0b1525', edgecolor='#1a2840', labelcolor='#c8d0e0')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Turbulence δB/B contour map

In [ ]:
Ip_grid  = np.linspace(0.5, 5.0, 60)
Pnbi_grid= np.linspace(0, 100, 60)
IP, PN   = np.meshgrid(Ip_grid, Pnbi_grid)
TURB     = np.array([[derivedParams(ip, 3.5, p, 4.0).turb
                       for ip in Ip_grid] for p in Pnbi_grid])

fig, ax = plt.subplots(figsize=(9, 6))
cf = ax.contourf(IP, PN, TURB, levels=20,
                 cmap='inferno', vmin=0, vmax=0.9)
ax.contour(IP, PN, TURB, levels=[0.3, 0.6, 0.85],
           colors=['#00d4ff'], linewidths=0.8, linestyles='--')
cbar = fig.colorbar(cf, ax=ax)
cbar.set_label('δB/B (turbulence amplitude)', color='#c8d0e0')
cbar.ax.yaxis.set_tick_params(color='#c8d0e0', labelcolor='#c8d0e0')
ax.set_xlabel('Plasma current Ip (MA)', color='#c8d0e0')
ax.set_ylabel('NBI power P_NBI (MW)', color='#c8d0e0')
ax.set_title('Turbulence map  (Bt=3.5 T, ne=4×10¹⁹ m⁻³)', color='#00d4ff')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## Stability summary at ITER operating point

In [ ]:
cases = [
    ('ITER baseline',     15.0, 5.3, 50, 10.0),
    ('Low-q risk',         4.0, 2.0, 20,  4.0),
    ('High-beta risk',     2.0, 1.5, 95,  1.5),
    ('Nominal sim point',  2.0, 3.5, 20,  4.0),
]
print(f'{'Case':<22} {'q':>6} {'β_N':>7} {'v_tor':>8} {'δB/B':>7}  Stability')
print('-' * 80)
for name, ip, bt, p, ne in cases:
    d = derivedParams(ip, bt, p, ne)
    print(f'{name:<22} {d.q:>6.2f} {d.beta_N:>7.2f} {d.v_tor:>8.1f} {d.turb:>7.3f}  {stability_summary(d)}')